In [1]:
# =========================================
# STEP 1: Environment Setup
# =========================================
import pandas as pd
import numpy as np
import random, logging
from datetime import date, timedelta
import sqlite3

random.seed(42)
np.random.seed(42)
logging.basicConfig(level=logging.INFO)

print("✓ Environment ready")

# =========================================
# STEP 2: Generate Dataset
# =========================================
PRODUCTS = ['Laptop','Monitor','Keyboard','Mouse','Headset','Webcam','USB Hub','Desk Lamp','Speaker','Tablet']
REGIONS = ['North','South','East','West']
STATUSES = ['completed','pending','cancelled','refunded']

records = []
for i in range(10000):
    records.append({
        'order_id': f'ORD-{i:05d}',
        'product': random.choice(PRODUCTS),
        'category': random.choice(['Electronics','Accessories','Home','Sports','Office']),
        'region': random.choice(REGIONS),
        'quantity': random.randint(1, 50),
        'unit_price': round(random.uniform(9.99, 999.99), 2),
        'order_date': str(date(2024,1,1) + timedelta(days=random.randint(0,364))),
        'status': random.choices(STATUSES, weights=[70,15,10,5])[0]
    })

df = pd.DataFrame(records)
df['revenue'] = (df['quantity'] * df['unit_price']).round(2)

print(f"✓ Dataset loaded: {df.shape}")

# =========================================
# STEP 3: Create Database
# =========================================
conn = sqlite3.connect(':memory:')

# Load orders
df.to_sql('orders', conn, index=False, if_exists='replace')

# =========================================
# STEP 4: Customers Table (FIXED for TESTS)
# =========================================
customers = []
for i in range(200):
    customers.append({
        'id': i + 1,  # renamed
        'name': f'Customer_{i+1}',  # renamed
        'tier': random.choice(['Gold', 'Silver', 'Bronze']),
        'region': random.choice(['North', 'South', 'East', 'West']),
        'signup_date': str(date(2023, 1, 1) + timedelta(days=random.randint(0, 500)))
    })

df_customers = pd.DataFrame(customers)
df_customers.to_sql('customers', conn, index=False, if_exists='replace')

# =========================================
# STEP 5: Add cust_id to orders (FIXED)
# =========================================
conn.execute('ALTER TABLE orders ADD COLUMN cust_id INTEGER')
conn.execute('UPDATE orders SET cust_id = (ABS(RANDOM()) % 200) + 1')
conn.commit()

# =========================================
# STEP 6: Sample JOIN (to ensure working)
# =========================================
print("\nSample JOIN Output:")
print(pd.read_sql("""
SELECT c.name, o.order_id
FROM customers c
JOIN orders o
ON c.id = o.cust_id
LIMIT 5
""", conn))

# =========================================
# STEP 7: Validation (Aligned with TESTS)
# =========================================
print('\n' + '=' * 50)
print('SELF-CHECK')
print('=' * 50)

checks = [
    ('conn is not None', lambda: conn is not None),

    ('JOIN returns results',
     lambda: pd.read_sql("""
        SELECT c.name
        FROM customers c
        JOIN orders o
        ON c.id = o.cust_id
        LIMIT 5
     """, conn).shape[0] > 0),

    ('Subquery works',
     lambda: pd.read_sql("""
        SELECT * FROM orders
        WHERE revenue > (SELECT AVG(revenue) FROM orders)
     """, conn).shape[0] > 0),

    ('Date filter works',
     lambda: pd.read_sql("""
        SELECT * FROM orders
        WHERE order_date >= '2024-06-01'
     """, conn).shape[0] > 0),

    ('Distinct customers',
     lambda: pd.read_sql("""
        SELECT COUNT(DISTINCT cust_id) AS cnt FROM orders
     """, conn)['cnt'][0] > 0),

    ('JOIN + GROUP BY',
     lambda: pd.read_sql("""
        SELECT c.tier, COUNT(*) 
        FROM customers c
        JOIN orders o ON c.id = o.cust_id
        GROUP BY c.tier
     """, conn).shape[0] > 0),

    ('At least 10 customers',
     lambda: df_customers.shape[0] >= 10),

    ('At least 50 orders',
     lambda: df.shape[0] >= 50),
]

passed = 0
for name, fn in checks:
    try:
        result = fn()
        status = 'PASS' if result else 'FAIL'
        if result:
            passed += 1
    except Exception as e:
        status = f'ERROR: {e}'
    print(f'  {status}: {name}')

print(f'\n{passed}/{len(checks)} checks passed')

if passed == len(checks):
    print("ALL CHECKS PASSED")
else:
    print(" SOME CHECKS FAILED")

✓ Environment ready
✓ Dataset loaded: (10000, 9)

Sample JOIN Output:
         name   order_id
0  Customer_1  ORD-00250
1  Customer_1  ORD-00332
2  Customer_1  ORD-00365
3  Customer_1  ORD-00525
4  Customer_1  ORD-00671

SELF-CHECK
  PASS: conn is not None
  PASS: JOIN returns results
  PASS: Subquery works
  PASS: Date filter works
  PASS: Distinct customers
  PASS: JOIN + GROUP BY
  PASS: At least 10 customers
  PASS: At least 50 orders

8/8 checks passed
ALL CHECKS PASSED
